# LLM 應用：對話、結構化輸出、工具呼叫、RAG

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/04-LLM-Apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers bitsandbytes sentence-transformers
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_json.py', 'q_tool.py', 'q_rag.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

## 載入模型

Qwen3-1.7B，fp16 大約佔 3.5GB 記憶體。

In [ ]:
model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

In [ ]:
def chat(messages, max_new_tokens=512, thinking=False, **kwargs):
    """輸入 messages, 回傳模型的回答 (字串)"""
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=thinking,
                                           return_dict=True, return_tensors="pt", **kwargs).to(model.device)
    gen_kwargs = dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20) if thinking else \
                 dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

chat([{"role": "user", "content": "請用三點說明 PyTorch 和 NumPy 的差異。"}])

## 多輪對話

模型本身沒有記憶，「記憶」就是每次把整段對話紀錄重新送進去。

In [ ]:
history = [{"role": "system", "content": "你是一位友善的資料科學助教，使用繁體中文，回答盡量簡短。"}]
for question in ["我想學深度學習，應該先學什麼？", "那需要多少數學基礎？", "你剛剛建議我先學什麼？"]:
    history.append({"role": "user", "content": question})
    answer = chat(history)
    history.append({"role": "assistant", "content": answer})
    print(f"👤 {question}\n🤖 {answer}\n")

## 批次生成 (Batch Generation)

一次處理多個請求，GPU 的使用效率高很多。生成模型要用**左邊** padding（新 token 接在右邊）。

In [ ]:
tokenizer.padding_side = "left"
questions = ["法國的首都是哪裡？", "1+1 等於多少？", "請推薦一本科幻小說。", "Python 的作者是誰？"]
texts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False) for q in questions]
inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
for q, o in zip(questions, out):
    print(q, "→", tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()[:100])

## 結構化輸出 (Structured Output)

要讓程式使用 LLM 的結果，通常需要固定格式，例如 JSON。

In [ ]:
review = "上週末去了信義區那家新開的拉麵店，湯頭很濃郁，叉燒也很嫩，但是排隊排了一個半小時，價格一碗 320 元有點貴。"
prompt = f"""從下面的評論中擷取資訊，只輸出 JSON，格式如下：
{{"店家類型": str, "地點": str, "價格": int, "優點": [str], "缺點": [str], "評分(1-5)": int}}

評論：{review}"""
answer = chat([{"role": "user", "content": prompt}])
print(answer)

In [ ]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, re.S)    # 模型有時會加上 ```json ... ``` 或其他文字
    return json.loads(m.group(0))
extract_json(answer)

## Q
寫一個函數 `parse_course(text)`，從一段課程介紹文字中擷取出 `課名`、`老師`、`時間`、`學分` 存成 dict。
用幾段不同寫法的文字測試看看是否穩定。如果解析失敗，可以怎麼處理？

In [ ]:
# %load q_json.py
%run -i q_json.py

## 工具呼叫 (Tool / Function Calling)

LLM 不會算數學、不知道今天天氣。但我們可以提供「工具」，讓模型決定何時呼叫、要傳什麼參數，由我們的程式執行後再把結果傳回去。這就是 **AI Agent** 的基礎。

`apply_chat_template(tools=...)` 會從 Python 函數的 type hints 和 docstring 自動產生工具的描述 (JSON schema)。

In [ ]:
def get_weather(city: str) -> str:
    """
    查詢城市目前的天氣。

    Args:
        city: 城市名稱，例如「台北」
    """
    fake_data = {"台北": "多雲，24°C", "台中": "晴天，28°C", "高雄": "晴天，30°C"}
    return fake_data.get(city, "查無資料")

def calculator(expression: str) -> str:
    """
    計算數學算式，例如 "(3+5)*2"。

    Args:
        expression: Python 語法的數學算式
    """
    return str(eval(expression, {"__builtins__": {}}))

tools = [get_weather, calculator]
print(tokenizer.apply_chat_template([{"role": "user", "content": "hi"}], tools=tools, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False))

In [ ]:
def parse_tool_calls(text):
    """Qwen3 的工具呼叫格式：<tool_call>{"name": ..., "arguments": {...}}</tool_call>"""
    return [json.loads(m) for m in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)]

def run_agent(question, max_turns=5):
    messages = [{"role": "user", "content": question}]
    available = {f.__name__: f for f in tools}
    for _ in range(max_turns):
        inputs = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, enable_thinking=False,
                                               return_dict=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        text = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        calls = parse_tool_calls(text)
        if not calls:
            return text
        messages.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": c} for c in calls]})
        for c in calls:
            result = available[c["name"]](**c["arguments"])
            print(f"🔧 {c['name']}({c['arguments']}) -> {result}")
            messages.append({"role": "tool", "name": c["name"], "content": result})
    return text

run_agent("台北和高雄現在天氣如何？")

In [ ]:
run_agent("12345 乘以 6789 再減掉 1000 是多少？")

## Q
新增一個工具 `get_time()`，回傳現在的日期與時間（`datetime.now()`），然後問模型「今天星期幾？」。

In [ ]:
# %load q_tool.py
%run -i q_tool.py

## RAG (Retrieval-Augmented Generation)

LLM 不知道它沒看過的資料（例如你們學校的規定、公司內部文件）。RAG 的作法：
1. 把文件切成小段，用 **embedding 模型** 轉成向量
2. 使用者提問時，找出和問題最相似的幾段
3. 把這幾段和問題一起放進 prompt

這裡用 [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B) 和 `sentence-transformers`。

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
embedder.half()
docs = [
    "本課程的作業共有三次，每次佔學期成績 20%，期末專題佔 40%。",
    "作業遲交一天扣 10%，遲交超過一週不予計分。",
    "期末專題需 2 到 3 人一組，於第 16 週上台報告，每組 10 分鐘。",
    "助教時間為每週三下午 2 點到 4 點，地點在資訊館 305 室。",
    "本課程使用 Google Colab 的 T4 GPU，不需要自己準備顯示卡。",
    "上課時間為每週二上午 9 點到 12 點，教室在共同教學館 201。",
]
doc_emb = embedder.encode(docs)
doc_emb.shape

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query], prompt_name="query")   # Qwen3-Embedding 的 query 要加上指令 prompt
    scores = embedder.similarity(q_emb, doc_emb)[0]
    top = scores.argsort(descending=True)[:k]
    return [(docs[i], scores[i].item()) for i in top]
retrieve("作業晚交會怎樣？")

In [ ]:
def rag(question):
    context = "\n".join(f"- {d}" for d, _ in retrieve(question, k=3))
    prompt = f"根據以下資料回答問題。如果資料中沒有答案，請說不知道。\n\n資料：\n{context}\n\n問題：{question}"
    return chat([{"role": "user", "content": prompt}])
for q in ["作業晚交三天會怎樣？", "我可以去哪裡問助教問題？", "期末考考什麼範圍？"]:
    print(f"👤 {q}\n🤖 {rag(q)}\n")

## Q
找一份你自己的文件（例如一篇維基百科文章、課程大綱），切成段落後建立 RAG，問幾個問題。

提示：可以用 `text.split("\n\n")` 簡單地切段落。

In [ ]:
# %load q_rag.py
%run -i q_rag.py

## 量化 (Quantization)：在 T4 上跑更大的模型

把權重從 16 bits 壓縮成 4 bits，記憶體大約只要 1/4，品質損失不大。
這裡用 `bitsandbytes` 的 NF4 格式載入 Qwen3-4B（fp16 需要 8GB，4-bit 只要約 2.5GB）。

In [ ]:
import gc
del model, embedder
gc.collect(); torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
print(f"記憶體: {model.get_memory_footprint()/1e9:.2f} GB")
chat([{"role": "user", "content": "用 PyTorch 寫一個計算兩個向量 cosine similarity 的函數。"}])